In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import html
import contractions
import nltk
import os

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import joblib

In [31]:
# text preprocessing objects
svd_model = joblib.load('../../pickle-object/svd.pkl')
tfidf_vectorizer = joblib.load('../../pickle-object/tfidf.pkl')
final_n_svs = joblib.load('../../pickle-object/final_n_svs.pkl')

# model prediction objects
X_columns = joblib.load('../../pickle-object/X_columns.pkl')
model = joblib.load('../../pickle-object/lasso.pkl')

In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_nltk_alpha_only(text):
    text = html.unescape(str(text))
    text = contractions.fix(text)

    tokens = word_tokenize(text.lower())

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words and len(word) > 2
    ]

    return " ".join(tokens)

In [5]:
def description_to_svd_vector(description):
    processed_text = preprocess_nltk_alpha_only(description)

    tfidf_vector = tfidf_vectorizer.transform([processed_text])

    svd_vector = svd_model.transform(tfidf_vector)[:, :final_n_svs]

    svd_vector_df = pd.DataFrame(
        svd_vector,
        columns=[f"SV_{i+1}" for i in range(final_n_svs)]
    )

    return svd_vector_df

In [6]:
def cluster(INPUT_HERE):
    pass
    return cluster, competition_score

In [7]:
def predict_output(vec):
    return model.predict(pd.DataFrame(data=vec, columns=X_columns).fillna(0))

In [8]:
def run_pipeline(description=None, bin=False, bin_method=round):
    if description == None:
        description = input('Input game description: ')
        
    svd_df = description_to_svd_vector(description)

    predicted_score = predict_output(svd_df)
    if bin:
        predicted_score = bin_method(predicted_score[0])
    # cluster, competition_score = cluster(svd_df)

    print(f"Predicted Score: {predicted_score}")

In [35]:
description = """
"The dice game with the magic dots"

This simple game's theme is from the "Maggie and the Ferocious Beast" franchise.

It is meant for 1-4 players from Ages 3 1/2 to 6.

It involves rolling a special dice: two sides have a 1 and get the roller one dot from the middle of the table; two sides have a 2 and get two dots; one side is a hat that gets a dot from another player; one side is a hat which loses a dot to the player on the left.

The winner is the first player to fill his or her board's monster with dots.

Ravensburger Game no. 23 137 9"""

run_pipeline(description, bin=True)

Predicted Score: 5
